In [0]:

# Seção de criação do catalogo e schemas
spark.sql("CREATE CATALOG if NOT EXISTS medallion") #Criação do Catalog
spark.sql("USE CATALOG medallion") # utilização do catalag
spark.sql("CREATE VOLUME if NOT EXISTS landing") #Criação do volume
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze") #Criação do schema bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS silver") #Criação do schema silver
spark.sql("CREATE SCHEMA IF NOT EXISTS gold") #Criação do schema gold

In [0]:
# Esse bloco tem a função de ingestão dos dados originados das planilhas
from pyspark.sql import functions as F # importação das funções do pyspark.sql

path = "/Volumes/medallion/default/landing/" # Caminho de acesso dos arquivos

def inject_csv(nome_arquivo: str, nome_tabela: str):
    # Leitura do arquivo csv, passo o nome do arquivo e o nome da tabela como referencia para que o arquivo seja encontrado e armazenado na bronze da forma correta
    
    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(path + nome_arquivo)
    ) # armazeno no DataFrame a leitura do arquivo csv que veio da landing, o header é pra informar que a primeira linha das tabelas são os nomes das colunas, inferSchema é pra manter o formato dos elementos

    df = df.withColumn("ingestion_datetime", F.current_timestamp())# Adiciono no Dataframe uma coluna com o tempo de inserção, lembrando que isso não altera a tabela original, apenas cria uma coluna no dataframe que será utilizado pra criar a tabela de bronze
    
    df.write.mode("append").format("delta").option("mergeSchema", True).saveAsTable(nome_tabela) # Escrevo no delta a tabela bronze, utilizando o metodo append exigido no documento
    

tabelas_csv = [
    ("movies_info_TMDB_IMDB.csv", "bronze.tb_movies_info"),
    ("movies_financials_IMDB_TMDB.csv", "bronze.tb_movies_financials"),
    ("movies_metrics_IMDB_TMDB.csv", "bronze.tb_movies_metrics"),
    ("credits_and_tags_IMDB_TMDB.csv", "bronze.tb_credits_and_tags"),
    ("movies_reviews.csv", "bronze.tb_movies_reviews"),
] # Coloquei o nome das tabelas csv e o nome da tabela que será criada na bronze, optei de colocar em uma lista para que eu rode o for e a ingestão seja automatizada

for nome_csv, nome_tabela in tabelas_csv:
    inject_csv(nome_csv, nome_tabela) # Rodando o for para rodar a função de ingestão

In [0]:
# Nesse bloco criamos uma função responsavel por realizar a requisição na API, e retornar um JSON para a função de criação de tabelas
import requests
from datetime import datetime
def requisicao_api(inicio: str, fim: str): # Recebemos como parametro a data de inicio e fim
    # Conversão para o formato exigido pela API: MM-DD-AAAA
    data_inicio = datetime.strptime(inicio, "%Y-%m-%d").strftime("%m-%d-%Y")
    data_fim = datetime.strptime(fim, "%Y-%m-%d").strftime("%m-%d-%Y")

    url = (
        f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,"
        f"dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
        f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"
    ) # Chamando a URL da API, passando os parametros de data inicio e fim,

    resposta = requests.get(url) # Chamando a requisição
    if resposta.status_code != 200:
        raise Exception("Não foi possível realizar a requisição") #verificação se foi encontrado algum erro na requisição

    data = resposta.json().get("value", [])
    return data #Retornamos o JSON

In [0]:

# Esse bloco tem a função de adicionar os elementos da API na tabela bronze

def criar_tabela_cotacao_dolar(dados: list): # Recebemos como parametro os dados da JSON
    if len(dados) == 0:
        print("Nenhuma cotação retornada para o período informado.")
        return # Verificando se houve retorno
    
    # Coversão de JSON para DF Spark
    df_cotacao = spark.createDataFrame(dados)

    # Adiciona a coluna de controle de ingestão, no mesmo padrão das outras tabelas bronze
    df_cotacao = df_cotacao.withColumn("ingestion_datetime", F.current_timestamp())
    df_cotacao.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("bronze.tb_cotacao_dolar") # adição tabela na bronze

    

In [0]:
from datetime import datetime, timedelta

hoje = datetime.today()

# Busca uma janela maior para garantir cotação anterior ao primeiro dia da Silver.
inicio_periodo = hoje - timedelta(days=10)

dbutils.widgets.text(
    "data_inicio",
    inicio_periodo.strftime("%Y-%m-%d")
)

dbutils.widgets.text(
    "data_fim",
    hoje.strftime("%Y-%m-%d")
)

inicio = dbutils.widgets.get("data_inicio")
fim = dbutils.widgets.get("data_fim")

dados = requisicao_api(inicio, fim)
criar_tabela_cotacao_dolar(dados)